# Flood event mapping using Sentinel-1 GRD data

Map the extent of flooded areas using Sentinel-1 data for pre and post event.

Generate a pre-event mosaic and use the first available image after the event. Calculate the ratio between pre and post event status and apply a threshold to get the area affected by flooding. 

In [18]:
import openeo
import rasterio
from openeo.processes import ProcessBuilder, array_element, quantiles, median
import rioxarray as rxr
import leafmap
from IPython.display import JSON
from shapely.geometry import shape
from folium.plugins import Draw
from scipy.ndimage import uniform_filter
import numpy as np

from pathlib import Path
import time

In [3]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [4]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany")
event = "Tuscany_2025_S1"

## 1) Define Area of Interest and dates of the event

In [5]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [6]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 10.699997, 'south': 43.716527, 'east': 10.866165, 'north': 43.835022}


In [7]:
PRE_DATE  = ("2025-01-01", "2025-03-10")   # before flood
POST_DATE = ("2025-03-15", "2025-03-20")   # during/after flood

## 2) Search stac catalog for items in the given time frame and area of interest

In [8]:
from pystac_client import Client
from datetime import datetime

catalog = Client.open("https://stac.dataspace.copernicus.eu/v1")

#for c in catalog.get_collections():
#    print(c.id)

In [9]:
# Search for Sentinel-1 GRD items
search = catalog.search(
    collections=["sentinel-1-grd"],   
    bbox=[bbox["west"], bbox["south"], bbox["east"], bbox["north"]],
    datetime=f"{PRE_DATE[0]}/{POST_DATE[1]}",
    
)
items = sorted(search.items(), key=lambda i: i.datetime)

## 3) Search for S1 GRD dates for pre and post event with matching orbit

In [10]:

from datetime import datetime, timezone

event = datetime(2025, 3, 14, tzinfo=timezone.utc)

items = sorted(items, key=lambda i: i.datetime)

# Split before and after event
pre_candidates = [
    i for i in items 
    if i.datetime < event
]

post_candidates = [
    i for i in items 
    if i.datetime > event
]

# Find a matching orbit with enough pre-event images
orbit_pair = None

for post in sorted(post_candidates, key=lambda i: i.datetime):
    #orbit_state = post.properties["sat:orbit_state"]
    relative_orbit = post.properties["sat:relative_orbit"]

    matching_pre = [
        pre for pre in pre_candidates
        if (
            #pre.properties["sat:orbit_state"] == orbit_state
            pre.properties["sat:relative_orbit"] == relative_orbit
        )
    ]

    if len(matching_pre) >= 2:  # require multiple images
        orbit_pair = (matching_pre, post)
        break

if orbit_pair is None:
    raise RuntimeError("No matching Sentinel-1 orbit with enough pre-event images found.")

pre_images, post = orbit_pair

# Sort pre-images chronologically
pre_images = sorted(pre_images, key=lambda i: i.datetime)

print("Selected pre-event images:")
for p in pre_images:
    print(p.datetime, p.properties["sat:orbit_state"], p.properties["sat:relative_orbit"])

print("\nSelected post-event image:")
print(post.datetime, post.properties["sat:orbit_state"], post.properties["sat:relative_orbit"])

Selected pre-event images:
2025-01-04 05:27:52.913896+00:00 descending 168
2025-01-16 05:27:52.031701+00:00 descending 168
2025-01-28 05:27:51.528541+00:00 descending 168
2025-02-09 05:27:51.119810+00:00 descending 168
2025-02-21 05:27:50.505862+00:00 descending 168
2025-03-05 05:27:50.496201+00:00 descending 168

Selected post-event image:
2025-03-17 05:27:50.792128+00:00 descending 168


## 4) Define final date window for pre event and post event 

In [11]:
pre_date = [pre_images[0].datetime.date(), pre_images[-1].datetime.date()]
pre_date = (pre_date[0].strftime("%Y-%m-%d"), pre_date[1].strftime("%Y-%m-%d"))
print(f"Pre-event date range: {pre_date} ")

post_date = (
    post.datetime.strftime("%Y-%m-%d"),
    post.datetime.strftime("%Y-%m-%d")
)

print(f"Post-event date: {post_date} ")

Pre-event date range: ('2025-01-04', '2025-03-05') 
Post-event date: ('2025-03-17', '2025-03-17') 


In [ ]:
#info = connection.describe_collection("SENTINEL1_GRD")
#print(info)

## 5) Load both data cubes

In [13]:
lee_udf = openeo.UDF("""
import numpy as np
import xarray as xr
from scipy.ndimage import uniform_filter

def lee(img, size, enl):
    img = img.astype(np.float32)
    valid = np.isfinite(img)
    filled = np.where(valid, img, 0.0)
    n = np.maximum(uniform_filter(valid.astype(np.float32), size, mode="reflect"), 1e-6)

    mean = uniform_filter(filled, size, mode="reflect") / n
    mean_sq = uniform_filter(filled ** 2, size, mode="reflect") / n
    var = np.maximum(mean_sq - mean ** 2, 0)

    noise_var = (mean ** 2) / enl                      
    w = np.maximum(var - noise_var, 0) / (var + 1e-10)

    out = mean + w * (img - mean)
    return np.where(valid, out, np.nan)

def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    size = context.get("size", 7)
    enl = context.get("enl", 4.4)
    out = xr.apply_ufunc(
        lee, cube,
        kwargs=dict(size=size, enl=enl),
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        vectorize=True,
    )
    return out.transpose(*cube.dims)
""")


def load_s1(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL1_GRD",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["VV", "VH"],
    ).sar_backscatter(coefficient="sigma0-ellipsoid")   
    return cube

def despeckle(cube, size=7, enl=4.4):
    return cube.apply_neighborhood(
        process=lee_udf,
        size=[{"dimension": "x", "value": 256, "unit": "px"},
              {"dimension": "y", "value": 256, "unit": "px"}],
        overlap=[{"dimension": "x", "value": 8, "unit": "px"},
                 {"dimension": "y", "value": 8, "unit": "px"}],
        context={"size": size, "enl": enl},
    )


def s1_composite(temporal_extent):
    filtered = despeckle(load_s1(temporal_extent))
    return filtered.reduce_dimension(dimension="t", reducer="mean")


pre_s1  = s1_composite(PRE_DATE)
post_s1 = s1_composite(POST_DATE)

# Log-ratio: PRE/POST in dB 
to_db = lambda c: c.apply(lambda x: 10 * x.log(base=10))
ratio_db = to_db(pre_s1).merge_cubes(to_db(post_s1), overlap_resolver="subtract")

ratio_db.create_job(title=f"S1_logratio_{event}", out_format="GTiff") \
        .start_and_wait().download_results(out_dir)

0:00:00 Job 'j-26100708035548a1b3e4c34a30b0f537': send 'start'
0:00:03 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:00:08 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:00:14 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:00:22 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:00:32 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:00:44 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:01:00 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:01:19 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:01:43 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:02:13 Job 'j-26100708035548a1b3e4c34a30b0f537': queued (progress 0%)
0:02:50 Job 'j-26100708035548a1b3e4c34a30b0f537': running (progress N/A)
0:03:37 Job 'j-26100708035548a1b3e4c34a30b0f537': running (progress N/A)
0:04:35 Job 'j-26100708035548a1b3e4c34a30b0f537': running (progress N/A)
0:05:36 

{PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany/openEO.tif'): {'eo:bands': [{'name': 'VV'},
   {'name': 'VH'}],
  'href': 'https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26100708035548a1b3e4c34a30b0f537/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=4c0785e574254212a1e4e958058f2d98%2F20261007%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20261007T080932Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYxMDA3MDgwMzU1NDhhMWIzZTRjMzRhMzBiMGY1MzciXSwidXNlcl9pZCI6WyI5N2JmZDdmMy1kMTQ1LTRlMDEtYjJkYS02NzEzZmRjNTJmZTgiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZC

### 7) Apply threshold to the result t0 get only the flooded areas

In [16]:
thresh = 5

change = (ratio_db >= thresh).apply(lambda x: x * 1) 

job = change.create_job(title=f"S1_change_{event}", out_format="GTiff")
job.start_and_wait()
job.download_result(Path(out_dir) / "S1_ratio_threshold_5.tif")

0:00:00 Job 'j-2610070847504b598dee54f18f451f4f': send 'start'
0:00:02 Job 'j-2610070847504b598dee54f18f451f4f': created (progress 0%)
0:00:07 Job 'j-2610070847504b598dee54f18f451f4f': queued (progress 0%)
0:00:14 Job 'j-2610070847504b598dee54f18f451f4f': queued (progress 0%)
0:00:22 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:00:31 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:00:44 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:00:59 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:01:18 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:01:42 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:02:12 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:02:50 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:03:36 Job 'j-2610070847504b598dee54f18f451f4f': running (progress N/A)
0:04:35 Job 'j-2610070847504b598dee54f18f451f4f': finished (progre

PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany/S1_ratio_threshold_5.tif')

### 8) Create Sentinel-2 outputs 

In [19]:
BANDS   = ["B03", "B11"]          # Green, SWIR1
BAD_SCL = [1, 3, 7, 8, 9, 10]

TIMESPAN = {
    "PRE": PRE_DATE,
    "POST": POST_DATE,
}

def load_s2(temporal_extent):
    bands = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="bilinear")  # B11 is 20 m -> 10 m

    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["SCL"],
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="near")   

    return bands, scl


def cloud_mask(scl):
    s = scl.band("SCL")
    m = s == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (s == v)
    return m


def masked_cube(temporal_extent):
    bands, scl = load_s2(temporal_extent)
    return bands.mask(cloud_mask(scl))


def composite_pre(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: array_element(quantiles(data=x, probabilities=[0.25]), index=0),
    )


def composite_post(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: median(x),
    )


def mndwi(mosaic):
    """MNDWI = (Green - SWIR1) / (Green + SWIR1) = (B03 - B11) / (B03 + B11)"""
    def _mndwi(b):
        b03 = b[0]
        b11 = b[1]
        return (b03 - b11) / (b03 + b11)
    return mosaic.reduce_dimension(dimension="bands", reducer=_mndwi)


COMPOSITES = {"PRE": composite_pre, "POST": composite_post}

jobs = {}
for name, extent in TIMESPAN.items():
    print(f"Starting {name}...")
    result = mndwi(COMPOSITES[name](extent)).save_result(format="GTiff")
    job = result.create_job(title=f"S2_MNDWI_{name}")
    job.start()
    jobs[name] = job
    print(f"{name} started: {job.job_id}")
    time.sleep(10)

Starting PRE...
PRE started: j-2610070911034ea982fc256e73571c67
Starting POST...
POST started: j-261007091117484ca61c3b4ac047b026


In [24]:
file_pre_mndwi = out_dir/ f"pre_Tuscany_MNDWI.tiff"
file_post_mndwi = out_dir/ f"post_Tuscany_MNDWI.tiff"

jobs["PRE"].get_results().download_file(file_pre_mndwi)
jobs["POST"].get_results().download_file(file_post_mndwi)

PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/tuscany/post_Tuscany_MNDWI.tiff')

### 9) Classify Sentinel-2 outputs 

In [31]:
thresh = 0.7

# 1 = low MNDWI (non-flooded), 0 = flooded, nodata stays nodata
with rasterio.open(file_pre_mndwi) as src_pre:
    pre_mndwi = src_pre.read(1).astype("float32")
    profile = src_pre.profile
    pre_nodata = src_pre.nodata

with rasterio.open(file_post_mndwi) as src_post:
    post_mndwi = src_post.read(1).astype("float32")
    post_nodata = src_post.nodata

# Valid-pixel mask: NaN or declared nodata must not be classified
valid = ~np.isnan(pre_mndwi) & ~np.isnan(post_mndwi)
if pre_nodata is not None:
    valid &= pre_mndwi != pre_nodata
if post_nodata is not None:
    valid &= post_mndwi != post_nodata

pre_low  = (pre_mndwi  <= thresh).astype("uint8")
post_low = (post_mndwi <= thresh).astype("uint8")

change_mndwi = (pre_low * 10) + post_low
change_mndwi[~valid] = 255  # nodata

file_change_mndwi = out_dir / f"change_Tuscany_MNDWI.tif"

profile.update(dtype="uint8", count=1, nodata=255, compress="lzw")
with rasterio.open(file_change_mndwi, "w", **profile) as dst:
    dst.write(change_mndwi, 1)

### 10) Combine Sentinel-1 and Sentinel-2 output

In [39]:
with rasterio.open(file_change_mndwi) as src:
    change_mndwi = src.read(1)
    print(f"Change MNDWI: {src.count} band(s), {src.width}x{src.height} pixels, nodata={src.nodata}")


with rasterio.open(Path(out_dir) / "S1_ratio_threshold_5.tif") as src:
    s1_change = src.read(1)
    print(f"S1 change: {src.count} band(s), {src.width}x{src.height} pixels, nodata={src.nodata}")


combined = np.zeros_like(change_mndwi, dtype="uint8")
combined[(change_mndwi == 10) & (s1_change == 1)] = 1

with rasterio.open(out_dir / f"combined_s1_s2_flood_tuscany.tif", "w", **profile) as dst:
    dst.write(combined, 1)

Change MNDWI: 1 band(s), 1366x1346 pixels, nodata=255.0
S1 change: 2 band(s), 1366x1346 pixels, nodata=nan
